<a href="https://colab.research.google.com/github/kader-xai/data-science-roadmap/blob/main/module_79_finops_for_ai_platforms.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

> 💰 **Module 79 — Cost / FinOps for AI Platforms** &nbsp;·&nbsp; part of [`data-science-roadmap`](https://github.com/kader-xai/data-science-roadmap)

> Phase 10 · Master-Plan Gaps


# Module 79 — FinOps for AI Platforms

> Two facts most ML curricula skip:
> 1. **The bill is the moat.** Frontier-lab budgets are measured in **billions of dollars**. A startup's cloud bill can dwarf its salaries within months of launching.
> 2. **The bill is the *easiest* lever you have.** Pre-deploy, every architecture choice (M44–M71) is also a $/Mtok choice. Post-deploy, **prompt caching, model routing, quantisation, batching, and spot capacity** can each cut cost by 30–80 %.
>
> This module is the disciplined view of money: how to measure it, attribute it, monitor it, and engineer it down without compromising the product.

### What you'll cover
1. The two cost engines — **training** vs **inference** — and how they differ
2. The **unit economics** of AI products — $/Mtok, $/request, gross margin
3. **Where the bill actually goes** — compute, storage, network, eval, labelling
4. **Hosted-API pricing** in 2026 + which tier wins per workload
5. **Inference cost levers** — caching, routing, quantisation, batching, speculation
6. **Training cost levers** — spot/preemptible, fp8, data efficiency, transfer learning
7. **Tagging + allocation + chargeback** — the boring FinOps that pays for itself
8. **Cost observability** — Langfuse / Helicone / vendor portals / Prometheus exporters
9. The 2026 real-number scoreboard — frontier training + inference costs
10. The 12-month FinOps playbook for an AI platform team


## 1 · The two cost engines

Every AI platform's bill is two giant line items:

```
                     ┌─────────────── TRAINING ───────────────┐
                     │  Pre-train: $10M – $1B+ per frontier   │
                     │  Fine-tune: $100s – $1M per cycle      │
                     │  Pattern: bursty, one-time, capex-like │
                     │  Owners: research, ML eng              │
                     └────────────────────────────────────────┘
                                       │
                                       ▼
                     ┌─────────────── INFERENCE ──────────────┐
                     │  $/request, billed every user action   │
                     │  Pattern: continuous, opex, scales w/ DAU│
                     │  Owners: platform, product eng         │
                     └────────────────────────────────────────┘
```

| | Training | Inference |
|---|---|---|
| Frequency | episodic | continuous |
| Predictability | known in advance | spiky with traffic |
| Optimisable by… | model size, data, parallelism (M66, M77) | caching, routing, quantisation (M38, M44, M71) |
| Failure mode | exceeded budget, missed launch | per-request margin erosion until you go broke |

You will get most of the early wins from **inference** optimisation — pre-train is dominated by a single line item (GPUs × time) and that's locked in once you launch.

## 2 · The unit economics

In [ ]:
# A sketch of what "knowing your unit economics" means in practice
def cost_per_request(
    input_tokens: int,
    output_tokens: int,
    in_price_per_mtok: float,    # $/MTok
    out_price_per_mtok: float,   # $/MTok
    cache_hit_rate: float = 0.0,
    cache_in_price_per_mtok: float = None,
) -> float:
    """Return USD cost per request."""
    cache_in_price = cache_in_price_per_mtok if cache_in_price_per_mtok is not None else in_price_per_mtok
    effective_in = (1 - cache_hit_rate) * in_price_per_mtok + cache_hit_rate * cache_in_price
    return (input_tokens  / 1e6) * effective_in + (output_tokens / 1e6) * out_price_per_mtok

# Example: a typical chatbot turn — 1.5k context, 500-token answer
costs = {
    "GPT-5 (full)":     (5.0,  10.0, 0.5,  0.5),    # made-up indicative numbers
    "Claude Sonnet 4.6":(3.0,   15.0, 0.3,  0.3),
    "Gemini 2.5 Pro":   (1.25,  10.0, 0.125, 0.125),
    "Llama 3.3 70B (self-host vLLM)": (0.4, 0.6, 0.0, 0.0),   # amortised GPU cost
}
for model, (i, o, ci, _) in costs.items():
    c = cost_per_request(1500, 500, i, o, cache_hit_rate=0.5, cache_in_price_per_mtok=ci)
    print(f"{model:<30s}  ${c*1000:.2f} / 1k requests")

**What the calculation reveals:** for a 50 % prompt-cache hit rate (M37 / M44 prefix cache), self-hosted Llama on vLLM is often **6-30×** cheaper than frontier hosted APIs — but the **quality gap** has to be acceptable for your use case. The whole job of a platform team is making that trade-off explicit, per route, per user tier.

## 3 · Where the bill actually goes

In [ ]:
bill_breakdown = '''
Typical breakdown for a $1M/yr AI platform (sub-frontier):
  COMPUTE (training + inference GPU/CPU)   55-70 %
  HOSTED LLM API (OpenAI/Anthropic/etc)    10-25 %
  STORAGE  (object + parallel FS)           5-10 %
  NETWORK egress + LB                       3-8  %
  HUMAN LABELLING + RLHF crews              2-10 %
  EVAL infrastructure                       1-3  %
  TOOLING (Datadog, Langfuse, etc.)         1-3  %
  ─────────────────────────────────────────
  TOTAL                                     100 %
'''
print(bill_breakdown)

**Two things that *don't* show up in this list and surprise people:**
- **Network egress** — 1¢/GB on AWS sounds tiny until you ship a 700 GB model checkpoint across regions for DR.
- **Human labelling** — $20-50/hr × thousands of hours for preference data (M62, M67) easily exceeds $500K/yr for a serious RLHF effort.

Frontier labs invert the ratios: their compute is ~85 % of total spend; human labour is < 5 %.

## 4 · Hosted-API pricing — the 2026 landscape

| Provider / Model | Input $/MTok | Output $/MTok | Cached input $/MTok | Best at |
|---|---|---|---|---|
| **OpenAI GPT-5 (Pro)** | $5 | $10 | $0.50 | hard reasoning, tool use |
| **OpenAI GPT-5 mini** | $0.30 | $1.20 | $0.03 | fast, cheap, agentic |
| **Anthropic Claude Sonnet 4.6** | $3 | $15 | $0.30 | long context, code, agents |
| **Anthropic Claude Haiku 4** | $0.80 | $4 | $0.08 | fast routing tier |
| **Google Gemini 2.5 Pro** | $1.25 | $10 | $0.125 | huge context (1M+ tokens) |
| **Google Gemini 2.5 Flash** | $0.075 | $0.30 | $0.0075 | cheapest hosted frontier |
| **Mistral Large 3 / Codestral** | $2 | $6 | — | EU sovereignty |
| **Self-hosted Llama 3.3 70B (vLLM)** | ~$0.4 amortised | ~$0.6 amortised | — | when volumes justify |
| **Self-hosted Qwen 3 32B / DeepSeek-V3** | ~$0.15-0.30 | ~$0.30-0.60 | — | very cost-sensitive |

> 🟡 **These numbers move every quarter.** Treat the table as a *shape* not a quote — check pricing pages before architecting. The constants you can rely on: **smaller-cheaper tiers exist for every provider**, and **prompt-cached input is ~10× cheaper than uncached** at almost every provider.

### Per-tier model routing
The single biggest cost lever a platform can pull:

```
                                ┌──────────── Router ────────────┐
   user message  ─────────────► │                                │
                                │  if simple_classify(msg)       │
                                │      → Haiku 4 / Gemini Flash  │  $0.075 - $0.30 / Mtok
                                │  elif tool_call_likely(msg)    │
                                │      → Claude Sonnet 4.6       │  $3 / $15
                                │  else (hard reasoning)         │
                                │      → GPT-5 Pro               │  $5 / $10
                                └────────────────────────────────┘
```

A "cheap-first" router that escalates only on low-confidence answers from the small model can cut cost **5-15×** with minimal user-visible quality loss. The whole prompt-engineering subfield of **LLM-as-classifier** exists to make `simple_classify` reliable.

## 5 · Inference cost levers

| Lever | Typical saving | When |
|---|---|---|
| **Prompt / prefix caching** (M37, M44) | 30-90% input cost | system prompts, RAG contexts shared across users |
| **Response caching** (Redis) | 50-99% on hits | repeated FAQs, deterministic answers |
| **Model routing** (above) | 5-15× on average | most chat apps have a long tail of simple queries |
| **Quantisation** (INT8 / FP8 / AWQ — M38, M71) | 2-4× throughput | self-hosted; tolerate ~1-2% quality drop |
| **Continuous batching + KV cache** (M44, M60) | 5-20× throughput | self-hosted any LLM |
| **Speculative decoding** (M44 §9, M71 §8) | 1.5-3× tok/sec | self-hosted; need a fast draft model |
| **Truncation + summarisation** of long context | proportional to tokens dropped | chat memory, RAG retrieval |
| **Smaller model + better prompting** | 10-50× | when frontier reasoning isn't needed |
| **Streaming + early stop** | 10-30% on output | UX still good; cut runaway responses |
| **Cheaper region / on-demand → spot** | 30-70% | tolerant to interruption |

> ⚠️ **The killer chart in any cost review.** Plot **cost per request** vs **median user satisfaction score** for each route. A 30-70% cost reduction with a 1-2% quality drop almost always wins. Without that chart, "let's just use GPT-5 for everything" wins by default and bankrupts you.

## 6 · Training cost levers

| Lever | Typical saving | When |
|---|---|---|
| **Spot / preemptible GPUs** | 50-80% off list price | tolerant to mid-run preemption; need checkpoint-from-Lustre (M78) |
| **Reserved / Savings Plans / 1-3yr commit** | 30-60% off | predictable steady-state inference fleet |
| **FP8 / BF16 mixed precision** (M66 §9) | 1.5-2× throughput | every modern training run |
| **Better data efficiency** | unbounded (cheaper data = cheaper training) | dedup + curriculum + retain only high-signal data |
| **Smaller distilled model + larger teacher** | huge | inference cost / quality target |
| **Skip pre-train, fine-tune** | 100-1000× cheaper | most enterprise use cases |
| **LoRA / QLoRA** (M39, M58 freezing) | 100× cheaper than full FT | most fine-tunes |
| **Resume from checkpoint** | proportional to wasted steps | every failed run |
| **Sequence packing + flash-attention** (M66 §9) | 2-3× throughput | every modern training run |

**The pretraining cheat code:** **don't pretrain**. 99 % of "we need a custom model" needs are RAG (M30) + SFT/DPO (M59 / M62) on top of an open base. Pre-training a frontier model is a $10M+ commitment with very specific business justifications.

## 7 · Tagging + allocation + chargeback

The boring FinOps that pays for itself within a quarter.

### Tag everything
Every cloud resource, every Helm release, every model deployment, every job:

```yaml
# In your Terraform / Helm / k8s manifests
labels:
  team: search
  env: prod
  product: chat
  cost_center: cc-1041
  owner: alice@example.com
  workload: inference        # or "training", "experiment", "eval"
```

| Tag | Used for |
|---|---|
| `team` / `cost_center` | per-team chargeback |
| `env` | dev/staging/prod budgets |
| `product` | which product is bleeding |
| `owner` | who to email when it bleeds |
| `workload` | training vs inference budgets |
| `model_id` / `model_version` | per-model unit economics |

### Allocate
Use **AWS Cost Allocation Tags** / **GCP Labels** / **Azure Resource Tags** to roll the bill up. Add **token-level metering** for hosted APIs (Langfuse, Helicone, LiteLLM all do this) so you can map a $20K OpenAI bill to a specific product feature.

### Chargeback or showback
- **Showback:** each team sees its cost, but doesn't get billed internally — softest.
- **Chargeback:** each team has a real internal budget and gets billed against it — hardest but most disciplined.
- **Tagging without enforcement = nothing changes.** Pair tags with monthly budget alerts in Slack.

## 8 · Cost observability — instrumentation

In [ ]:
# A minimal request-level cost tracker — every LLM call goes through this wrapper.
import time, uuid
PRICES = {
    "gpt-5":          (5.0,  10.0, 0.5),
    "gpt-5-mini":     (0.30, 1.20, 0.03),
    "claude-sonnet-4-6":(3.0,  15.0, 0.30),
    "claude-haiku-4": (0.80, 4.00, 0.08),
    "gemini-2.5-pro": (1.25, 10.0, 0.125),
    "gemini-2.5-flash":(0.075,0.30, 0.0075),
}

def track_llm_call(model, prompt_tokens, completion_tokens, cached_in=0,
                   user_id=None, route="default", trace_id=None):
    in_p, out_p, cached_p = PRICES[model]
    cost = ((prompt_tokens - cached_in) / 1e6 * in_p
            + cached_in            / 1e6 * cached_p
            + completion_tokens    / 1e6 * out_p)
    record = {
        "ts": time.time(),
        "trace_id": trace_id or str(uuid.uuid4()),
        "model": model,
        "tokens_in": prompt_tokens,
        "tokens_in_cached": cached_in,
        "tokens_out": completion_tokens,
        "usd": round(cost, 6),
        "user_id": user_id,
        "route": route,
    }
    # In real life: send to OTel (M51) + Langfuse + your warehouse
    print(record)
    return cost

# example
track_llm_call("gpt-5-mini", 1500, 500, cached_in=1000, user_id="u_42", route="search.summary")

**Tools that do this for you in 2026:**
- **Langfuse / Helicone / LangSmith / Arize Phoenix** — drop-in proxies that meter every call and attach `trace_id`, model, cost, latency. Pair with M51 OTel.
- **LiteLLM** — model router with built-in budget enforcement (max $/user, per-route).
- **AWS Cost Explorer / Anomaly Detection · GCP Recommender · Azure Cost Management** — vendor portals.
- **Vantage · CloudHealth · Cloudability · Apptio** — third-party FinOps.
- **OpenCost** — k8s-native cost allocation; pairs with Prometheus (M50).
- **dcgm-exporter + custom metrics** (M50) — GPU-utilisation × time × cost — proves which jobs are wasting silicon.

The North-Star dashboard for every AI platform: **$/Mtok served**, **$/active user / day**, **gross margin per product** — all sliced by `team` × `model_id`. If you can't draw that chart in 5 minutes, FinOps isn't happening.

## 9 · 2026 real-number scoreboard

### Public reported / estimated training costs

| Model | Estimated training cost |
|---|---|
| GPT-3 (2020) | ~$5M |
| GPT-4 | ~$60-80M |
| Gemini Ultra (2024) | ~$190M |
| Claude 3.5 Sonnet (2024) | ~$50-100M |
| Llama 3 405B (2024) | ~$60-90M |
| DeepSeek-V3 (2024) | ~**$5.6M** (efficiency win) |
| Frontier 2025-26 runs | $500M – $1B+ |

### Sample real-world inference numbers

| Use case | Order of magnitude |
|---|---|
| GPT-5 chat session, 50 turns, ~1k tokens each | $0.25 - $1.50 |
| Self-hosted Llama 3.3 70B on 8× H100, same session | $0.02 - $0.10 |
| 1B-token batch summarisation, Gemini Flash | ~$75 |
| 100M-image vision pipeline, Claude 3.5 Sonnet | $20K - $80K |
| ChatGPT free tier — per active user, per day | ~$0.10-1.00 (reported estimates) |
| Anthropic enterprise customer (large coding org) | $1M-50M/yr ranges |

The order-of-magnitude gaps between routes are *enormous*. **A platform that routes well costs the same as a much smaller one that doesn't.**

## 10 · The 12-month FinOps playbook

### Month 0-1 — Instrument
- Pick a metering layer (**Langfuse** or **Helicone** for LLM calls; **OpenCost** for k8s; **OTel** for traces).
- Tag every resource (`team / env / product / cost_center / owner / workload / model_id`).
- Wire up a **single dashboard** with $/day, $/Mtok, gross margin per route.

### Month 2-3 — Quick wins
- Turn on **prompt caching** everywhere (M37 / M44 / hosted API features).
- Implement a **cheap-first router** (Haiku / Flash / GPT-mini for "simple", escalate for "hard").
- Cap **max_tokens** on every call to a reasonable ceiling. Enable **streaming + early-stop**.
- Move stateless workers to **spot / preemptible** with checkpoint-from-Lustre (M78).

### Month 4-6 — Self-host where it pays
- Identify the top 3 routes by spend. For each, **benchmark Llama 3.3 70B / Qwen 3 32B on vLLM (M44) or Triton (M71)** vs. the hosted alternative. Migrate where margin justifies + quality survives.
- Stand up **FP8 / INT8 quantisation** (M38) — almost always free quality.
- **Continuous batching + KV cache** if not already (M44 / M60).

### Month 7-9 — Budgets + governance
- Implement **per-team budgets** in LiteLLM / your gateway with hard caps.
- Set up **monthly cost reviews** with the product PM for each route.
- **Spike alerts**: 1.5× moving-average → Slack + page.
- **Forecast** the next quarter's run-rate; reconcile against revenue.

### Month 10-12 — Strategic
- Pre-commit (Savings Plans, Reserved, Capacity Reservations) on the steady-state inference fleet.
- Negotiate enterprise contracts with hosted providers (10-30 % is common at $100K+/mo).
- Evaluate the **bare-metal vs cloud** crossover (M78) for your actual usage.
- Build the **margin-per-route** Slack bot the PMs check before approving features.

## ✅ Recap
- AI cost = **training** (episodic, capex-like) + **inference** (continuous, opex-like). Inference is where you get most early wins.
- The single biggest lever is **model routing** — push the long tail to a Haiku / Flash / mini tier.
- Prompt caching, response caching, quantisation, batching, and speculative decoding each unlock another 30-80%.
- **Tag everything** (team / env / product / workload / model_id) and **chargeback** — without that, FinOps isn't happening.
- Use **Langfuse / Helicone / OpenCost / OTel** for token-level cost attribution. Build the **$/route × $/team × margin** dashboard before you need it.
- Frontier training is **$50M – $1B+**; for everyone else, **don't pretrain** — RAG + SFT + DPO is 100-1000× cheaper.
- Run the **12-month playbook**: instrument → quick wins → self-host where it pays → budgets → strategic commits.

Next: **M80 — Multi-Tenant AI Platform Patterns**.
